# Week 7: How old are authors when they are first nominated?

Is the Giller a prize for established writers late in their careers, or does it find new voices? One way to approach this question is to ask **how old authors were when they were first nominated**. We can also compare those ages with the ages of authors when they **win**.

Our spreadsheet has the information we need: the `Year` of each nomination, each author's date of birth in the `DOB` column, and whether the book won in the `Status` column. But, as usual, the data needs some work first. In this notebook we will:

1. find each author's **first** nomination,
2. separate out the **year of birth** from the date of birth,
3. calculate each author's **age** at their first nomination, and
4. compare those ages with the ages of authors when they win.

This notebook builds on `pandas_intro.ipynb`, so we'll use `read_csv`, picking columns, filtering, and `.plot` without explaining them again.

This time we need one row per **book**, because each book has its own nomination year and status. So we load `giller_nominees_clean.csv` rather than `giller_authors.csv`, and keep only the five columns we need.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

books = pd.read_csv("giller_nominees_clean.csv")
books = books[["Status", "Year", "Title", "Author", "DOB"]]

print(len(books), "nominated books")
books.head(10)

Picking several columns works just like picking one, except that we give a *list* of column names, which is why there are two sets of square brackets.

## 2. Looking at the dates of birth

Look at the `DOB` column above. The dates are not all written the same way:
- a full date, like `09/06/1959` (day/month/year);
- just a year, like `1979`;
- a `-` when the date of birth is unknown.

Before we try to use a column, it's always worth checking for surprises. Here are two authors whose dates look different from the rest:

In [ ]:
books[(books["Author"] == "Paulette Jiles") | (books["Author"] == "Emma Donoghue")]

(Here `|` means "or": keep a row if *either* condition is true.)

Values like `04-Apr-43` and `Oct-69` are a common spreadsheet problem: Excel decided these cells looked like dates and changed how they were written, throwing away the first two digits of the year! We know from other rows and other sources that Paulette Jiles was born on April 4, 1943, and that Emma Donoghue's other rows say `10/1969`.

### Fixing values with a dictionary

We've seen this pattern before: a dictionary records our corrections, with the messy value as the key and the corrected value as the value. pandas' `.replace()` method applies the whole dictionary to a column at once.

In [ ]:
dob_corrections = {
    "04-Apr-43": "04/04/1943",   # Paulette Jiles
    "Oct-69": "10/1969",         # Emma Donoghue
}

books["DOB"] = books["DOB"].replace(dob_corrections)

# Keep the birth year as a column so we can reuse it later.
books["Birth year"] = books["DOB"].str[-4:]

books[(books["Author"] == "Paulette Jiles") | (books["Author"] == "Emma Donoghue")]

## 3. Each author's first nomination

Many authors were nominated more than once, but we want the age at their **first** nomination. We can find it in two steps:

1. **Sort** the books by year with `.sort_values("Year")`, so every author's earliest book comes before their later ones.
2. **Keep the first row for each author** with `.drop_duplicates("Author")`. When an author appears more than once, it keeps the first appearance, which, after sorting, is their earliest nomination, and drops the rest.

In [ ]:
books_by_year = books.sort_values("Year")
first_nominations = books_by_year.drop_duplicates("Author")

print(len(first_nominations), "authors")
first_nominations.head()

Let's check that it worked for an author with several nominations. Emma Donoghue was nominated in 2008, 2016, and 2020:

In [ ]:
first_nominations[first_nominations["Author"] == "Emma Donoghue"]

## 4. Keeping authors with a known birth year

After correcting the date values, we added a `Birth year` column to `books` by taking the last four characters of each `DOB`. The year is now available on every book row, so it will come along when we select first nominations or winners.

Some birth years are still unknown. We'll leave those rows out and report how many there are. Then we convert the text in `Birth year` to integers so we can subtract it from a year.

In [ ]:
known = first_nominations[first_nominations["Birth year"] != "-"].copy()

print(len(known), "authors with a known date of birth")
print(len(first_nominations) - len(known), "authors left out because their date of birth is unknown")

We created the `Birth year` column by taking the last four characters of each `DOB`. The slice works the same way on a single string:

In [ ]:
dob = "09/06/1959"
print(dob[-4:])

To apply that slice to every value, we use `.str[-4:]`. We stored the result in `books["Birth year"]`, so it is already present in `first_nominations` and will also be available when we select winner rows. The values are text, so we convert the known birth years to integers before subtracting.

In [ ]:
known["Birth year"] = known["Birth year"].astype(int)

known.head()

## 5. Age at first nomination

Subtracting one column from another subtracts the values row by row, so one line calculates every author's age.

This age is approximate: we know the *year* of the nomination but not the exact day, so an author may have been a year younger than our calculation says. For a question about careers, that's close enough.

In [ ]:
known["Age"] = known["Year"] - known["Birth year"]

known.head()

Who were the youngest and oldest first-time nominees? Sorting by `Age` puts the youngest first. `.head()` shows the first rows (the youngest), and `.tail()` shows the last rows (the oldest).

In [ ]:
known_by_age = known.sort_values("Age")

known_by_age[["Author", "Year", "Age"]].head(5)   # the youngest

In [ ]:
known_by_age[["Author", "Year", "Age"]].tail(5)   # the oldest

In Week 6 we learned that the **median**, the value in the middle when everything is lined up in order, describes a "typical" value well, especially when a few unusual values could pull the average up or down. pandas calculates both for us.

In [ ]:
print(f"Median age at first nomination:  {known['Age'].median():.0f}")
print(f"Average age at first nomination: {known['Age'].mean():.1f}")

## 6. A chart of ages: the histogram

For a column of numbers like ages, the natural chart is a **histogram**. It divides the ages into ranges, called **bins** (here, 5 years wide: 25–29, 30–34, and so on), and draws one bar for each bin showing how many authors fall into it.

`bins=range(25, 90, 5)` sets where the bins start: 25, 30, 35, ... up to 85. We draw a vertical line at the median, so a reader can see where the middle is.

In [ ]:
median_age = known["Age"].median()

ax = known["Age"].plot.hist(bins=range(25, 90, 5), color="#2a78d6", edgecolor="white")

ax.axvline(median_age, color="black", linewidth=1)
ax.text(median_age + 1, ax.get_ylim()[1] * 0.92, f"Median: {median_age:.0f}")

ax.set_title("The typical author is first nominated for the Giller in their 40s")
ax.set_xlabel("Age at first Giller nomination")
ax.set_ylabel("Number of authors")
plt.show()

(`ax.get_ylim()[1]` is the top of the chart's vertical axis. Multiplying it by 0.92 places the label just below the top.)

**Discussion:**
- Is the Giller a prize for established writers or for new voices? How does this chart help you argue either side? What would a first *novel* have to do with it?
- We left out 50 authors whose date of birth is unknown. Which authors are most likely to be missing a date of birth from Wikipedia and other sources? Could leaving them out make the typical age look older or younger than it really is?
- The oldest first-time nominee, Anne Hébert, was nominated in 1999 for an English *translation* of a novel she first published in French. How might translation affect the age at which an author is nominated for the Giller?
- From the Week 6 notes: the prize changed from a shortlist of about 5 books to a longlist of 12 or more partway through. Would you expect that to change the ages of first-time nominees?

## 7. How old are authors when they win?

Now let's compare first-nomination ages with **winning** ages. The `Status` column marks winning books as `winner` or `winner (tie)`. We'll include ties. Each winning book is one row, so an author who wins more than once will appear more than once.

The `Birth year` column is already part of `books`, so winner rows have it too. We can filter out rows with an unknown birth year, convert the remaining years to integers, and calculate the ages.

In [ ]:
winner_rows = books[books["Status"].isin(["winner", "winner (tie)"])].copy()
winners_known = winner_rows[winner_rows["Birth year"] != "-"].copy()
winners_known["Birth year"] = winners_known["Birth year"].astype(int)
winners_known["Age"] = winners_known["Year"] - winners_known["Birth year"]

print(len(winner_rows), "winning books")
print(len(winners_known), "winning books with known dates of birth")
print(len(winner_rows) - len(winners_known), "winning books left out")

winners_known[["Year", "Title", "Author", "Age"]].sort_values("Year")

In [ ]:
median_winner_age = winners_known["Age"].median()

print(f"Median age when winning:  {median_winner_age:.0f}")
print(f"Average age when winning: {winners_known['Age'].mean():.1f}")

ax = winners_known["Age"].plot.hist(
    bins=range(25, 80, 5), color="#2a78d6", edgecolor="white"
)

ax.axvline(median_winner_age, color="black", linewidth=1)
ax.text(
    median_winner_age + 1,
    ax.get_ylim()[1] * 0.92,
    f"Median: {median_winner_age:.0f}",
)

ax.set_title("How old are authors when they win the Giller?")
ax.set_xlabel("Age when winning the Giller")
ax.set_ylabel("Number of winning books")
plt.show()

### Are winners getting older or younger over time?

A scatter plot lets us inspect the ages of winners by award year. A simple linear regression summarizes the overall direction: its slope estimates the average change in winning age per year. We can multiply the slope by 10 to describe the change per decade. This is an exploratory summary, not proof of a trend; there are few winners, and some authors won more than once.

In [ ]:
from scipy.stats import linregress

trend = linregress(winners_known["Year"], winners_known["Age"])

print("Winning books with known ages:", len(winners_known))
print(f"Estimated change per decade: {trend.slope * 10:.1f} years")
print(f"Pearson correlation: {trend.rvalue:.2f}")
print(f"p-value for a linear trend: {trend.pvalue:.3f}")

fig, ax = plt.subplots()
ax.scatter(
    winners_known["Year"],
    winners_known["Age"],
    color="#d62728",
    label="Winning books",
)
y_line = trend.slope * winners_known["Year"] + trend.intercept
ax.plot(winners_known["Year"], y_line, color="black", label="Linear trend")

ax.set_title("Winning authors' ages over time")
ax.set_xlabel("Year of award")
ax.set_ylabel("Author age in award year")
ax.legend()
plt.show()

In this dataset, the fitted line slopes downward by about **4.6 years per decade**. The p-value is about **0.053**, so this small sample gives suggestive evidence of a downward linear trend, but not strong evidence by the usual 0.05 cutoff. This does not prove that there is no trend. Also, authors who won multiple times contribute multiple rows, so the simple regression's p-value should be treated cautiously; the result describes the changing ages of winners as a group, not individual authors becoming younger.

**Discussion:** Compare the median age at first nomination with the median age when winning. Are winners older, younger, or about the same age as first-time nominees? Remember that winners are a selected group: winning an award is not the same as being nominated.

### All nomination ages, by list status

The chart below includes every nominated book with a known birth year. Each dot represents one book nomination, not a unique author; repeat nominees contribute multiple dots. The colors and marker shapes distinguish short-listed books, long-listed books, and winners (including ties).

In [ ]:
all_known = books[books["Birth year"] != "-"].copy()
all_known["Birth year"] = all_known["Birth year"].astype(int)
all_known["Age"] = all_known["Year"] - all_known["Birth year"]

short_listed = all_known["Status"] == "short-listed"
long_listed = all_known["Status"] == "long-listed"
winners = all_known["Status"].isin(["winner", "winner (tie)"])

fig, ax = plt.subplots()
ax.scatter(
    all_known.loc[short_listed, "Year"],
    all_known.loc[short_listed, "Age"],
    color="#2a78d6",
    marker="o",
    alpha=0.55,
    label="Short-listed",
)
ax.scatter(
    all_known.loc[long_listed, "Year"],
    all_known.loc[long_listed, "Age"],
    color="#e68613",
    marker="^",
    alpha=0.75,
    label="Long-listed",
)
ax.scatter(
    all_known.loc[winners, "Year"],
    all_known.loc[winners, "Age"],
    color="#d62728",
    marker="*",
    s=110,
    label="Winner",
    zorder=3,
)

ax.set_title("Author ages for all Giller nominations")
ax.set_xlabel("Year of nomination")
ax.set_ylabel("Author age in nomination year")
ax.legend()
plt.show()

## 8. Try it yourself

1. Change the bins in either histogram to be 10 years wide. Does the chart tell the same story? Which bin width do you prefer, and why?
2. Use filtering to compare the median age of first-time nominees **before 2006** with the median age **from 2006 on**. Are first-time nominees getting younger, older, or staying about the same?
3. Make a list of the authors who were first nominated before the age of 35. Do you recognize any of their names?
4. **Challenge:** A **scatter plot** draws one dot per row, using two numeric columns. Try `known.plot.scatter(x="Year", y="Age")`. What pattern, if any, do you see? Add a title and axis labels.
5. Compare the ages of winners before 2006 with the ages of winners from 2006 on. Does the typical winning age appear to change?

## Possible solutions

Here are one set of possible solutions. Try each question before looking at its code.

### 1. Histograms with 10-year bins

In [ ]:
ax = known["Age"].plot.hist(
    bins=range(20, 100, 10), color="#2a78d6", edgecolor="white"
)
ax.set_title("Age at first nomination (10-year bins)")
ax.set_xlabel("Age at first Giller nomination")
ax.set_ylabel("Number of authors")
plt.show()

ax = winners_known["Age"].plot.hist(
    bins=range(20, 100, 10), color="#2a78d6", edgecolor="white"
)
ax.set_title("Age when winning (10-year bins)")
ax.set_xlabel("Age when winning the Giller")
ax.set_ylabel("Number of winning books")
plt.show()

### 2. Compare median ages of first-time nominees

In [ ]:
before_2006 = known[known["Year"] < 2006]
from_2006 = known[known["Year"] >= 2006]

print("Median before 2006:", before_2006["Age"].median())
print("Median from 2006 on:", from_2006["Age"].median())

### 3. Authors first nominated before age 35

In [ ]:
under_35 = known[known["Age"] < 35]
under_35[["Author", "Year", "Age"]].sort_values("Age")

### 4. Scatter plot of first-nomination year and age

In [ ]:
ax = known.plot.scatter(x="Year", y="Age", color="#2a78d6")
ax.set_title("Age at first Giller nomination over time")
ax.set_xlabel("Year of first Giller nomination")
ax.set_ylabel("Age at first nomination")
plt.show()

### 5. Compare median ages of winners

In [ ]:
winners_before_2006 = winners_known[winners_known["Year"] < 2006]
winners_from_2006 = winners_known[winners_known["Year"] >= 2006]

print("Median winner age before 2006:", winners_before_2006["Age"].median())
print("Median winner age from 2006 on:", winners_from_2006["Age"].median())